In [27]:
import pandas as pd
import numpy as np
import json
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
import re
import os
from dotenv import load_dotenv

load_dotenv()

snowflake_connection_string = os.getenv('connection_string')

In [2]:

parts = snowflake_connection_string.split("//")[1].split("/")  
account = ".".join(parts[0].split('.')[:2]) 
user = parts[1].split('user=')[1].split('&')[0] 
password = parts[1].split('password=')[1].split('&')[0] 
database = parts[1].split('db=')[1].split('&')[0] 
warehouse = parts[1].split('warehouse=')[1].split('&')[0]  
role = parts[1].split('role=')[1]

engine = create_engine(URL(
    account = account,
    user = user,
    password = password,
    database = database,
    schema = 'gst_curated',
    warehouse = warehouse,
    role = role
))
cur = engine.connect()

def read_data_from_snowflake_table(cur,query):
    df = pd.read_sql(query, cur)
    return df


In [28]:
polymers = read_data_from_snowflake_table(cur,"select distinct polymer from SPT")
polymers = [re.sub(r'\W+', '', polymer.lower()) for polymer in polymers['polymer']]

In [67]:
# filler_lookup = read_data_from_snowflake_table(cur,"select * from gst_stg.filler_lookup_data")
filler_lookup = pd.read_csv("Filler Lookup Data.csv")
fillers = [filler.lower() for filler in filler_lookup['Filler Shortname'] if len(filler) in [2,3]]

In [70]:
f"({'|'.join(polymers)})({'|'.join(fillers)})"

'(pps|pehmw|pa610|pa66|tpv|pa666|tpe|tpc|pa1010|pa6t6i|pp|pa666t|pet|ppa|pehd|lcp|abs|pa6|pom|pa612|pa6t66|pa|tpu|peuhmw|pc|pct|pbt|pa6txt)(cf|cd|gf|gb|gd|gx|gm|mf|md|mx|nf|af|mh|lgf|laf|lcf)'

In [85]:
polymer_filler_pattern= r"^(pps|pehmw|pa610|pa66|tpv|pa666|tpe|tpc|pa1010|pa6t6i|pp|pa666t|pet|ppa|pehd|lcp|abs|pa6|pom|pa612|pa6t66|pa|tpu|peuhmw|pc|pct|pbt|pa6txt)-(cf|cd|gf|gb|gd|gx|gm|mf|md|mx|nf|af|mh|lgf|laf|lcf)(|\d{1}|\d{2})($)"
    

In [87]:
re.match(polymer_filler_pattern, "pa66-lcf")

<re.Match object; span=(0, 8), match='pa66-lcf'>